# Wave 1 LoRA-LLR canary

在 Colab 将硬件加速器设为 **L4 GPU**，然后运行下方唯一的代码单元。该单元从 GitHub 克隆 `origin/main`，记录实际 SHA，只跑 `GFP_AEQVI_Sarkisyan_2016 / modulo / fold 0 / lora_llr`（ESM-2 650M，N=256，seed=42）。GitHub 的 `main` 需先包含阶段 A 的 Parquet runner。

克隆、依赖安装、数据与模型下载和运行共用约 **12 分钟**的预算。成功 ZIP 包含单折 JSON、Parquet、日志和运行元数据。失败 ZIP 包含日志和运行元数据。下载弹窗未出现时，在 Colab 文件面板按输出中的路径下载。

确认 ZIP 已保存到本机 D 盘后，在 **运行时 → 断开连接并删除运行时** 释放 GPU，再将 ZIP 交回发起任务的线程。

In [ ]:
import hashlib
import json
import os
import shlex
import subprocess
import sys
import tempfile
import time
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

from google.colab import files

started = time.monotonic()
started_utc = datetime.now(timezone.utc).isoformat()
root = Path(tempfile.mkdtemp(prefix="wave1_lora_llr_canary_", dir="/content"))
repo = root / "repo"
work = root / "work"
out = root / "outputs"
log = root / "run.log"
metadata = root / "run_metadata.json"
repo_url = "https://github.com/wl2116641973-dev/esm2-fitness-ood.git"
protein = "GFP_AEQVI_Sarkisyan_2016"
model_size = "650M"
batch_size = 8
runner = repo / "scripts" / "run_wave1_lora_colab.py"
args = [
    "--proteins", protein, "--splits", "modulo", "--folds", "0",
    "--methods", "lora_llr", "--model-size", model_size, "--batch-size", str(batch_size),
    "--work-dir", str(work), "--out-dir", str(out),
]
packages = ("torch", "transformers", "peft", "accelerate", "pyarrow", "pandas", "numpy", "scipy")
info = {
    "status": "failed", "started_utc": started_utc, "origin": repo_url,
    "ref": "origin/main", "git_commit": None, "runner_args": args,
    "python": sys.version, "time_budget_sec": 720,
}
(root / "tmp").mkdir()
env = {
    **os.environ, "GIT_TERMINAL_PROMPT": "0", "HF_HOME": str(root / "hf"),
    "PYTHONPATH": str(repo / "src"), "PYTHONDONTWRITEBYTECODE": "1",
    "TMPDIR": str(root / "tmp"),
}

def run(*cmd, capture=False):
    # Reserve 15 seconds for termination and creating the ZIP.
    remaining = 705 - (time.monotonic() - started)
    if remaining <= 0:
        raise TimeoutError("Canary command budget exhausted.")
    print("$", shlex.join(cmd), flush=True)
    with log.open("a", encoding="utf-8") as handle:
        print("$", shlex.join(cmd), file=handle, flush=True)
        result = subprocess.run(
            ["timeout", "--kill-after=5s", f"{remaining:.3f}s", *cmd],
            cwd=root, env=env, stdin=subprocess.DEVNULL, stderr=subprocess.STDOUT,
            stdout=subprocess.PIPE if capture else handle, text=True,
        )
        if capture:
            print(result.stdout, file=handle, end="")
        print(f"exit_code={result.returncode}", file=handle)
    if result.returncode == 124:
        raise TimeoutError(f"Command reached the canary deadline: {cmd[0]}")
    result.check_returncode()
    return result.stdout.strip() if capture else None

artifacts = []
try:
    gpu = run("nvidia-smi", "--query-gpu=name,driver_version,memory.total",
              "--format=csv,noheader", capture=True)
    info["gpu"] = gpu
    if len(gpu.splitlines()) != 1 or gpu.split(",")[0].strip() != "NVIDIA L4":
        raise RuntimeError(f"Switch the Colab runtime to a single NVIDIA L4. Found: {gpu}")
    run("git", "clone", "--depth", "1", "--branch", "main", repo_url, str(repo))
    commit = run("git", "-C", str(repo), "rev-parse", "origin/main", capture=True)
    info["git_commit"] = commit
    print(f"origin/main = {commit}", flush=True)
    if run("git", "-C", str(repo), "rev-parse", "HEAD", capture=True) != commit:
        raise RuntimeError("Cloned HEAD differs from origin/main.")
    if not (repo / "src" / "esm2_ood" / "predictions.py").is_file() or not runner.is_file():
        raise RuntimeError("origin/main must contain the Stage A runner and predictions module.")
    source = runner.read_text(encoding="utf-8")
    required = ("write_predictions(", '"prediction_sha256"', *args[::2])
    if any(marker not in source for marker in required):
        raise RuntimeError("origin/main runner lacks the Stage A single-fold Parquet contract.")

    pip = (sys.executable, "-m", "pip", "--disable-pip-version-check", "--no-input")
    run(*pip, "uninstall", "-y", "torchao")
    run(*pip, "install", "--no-cache-dir", *packages[1:])
    info["packages"] = {name: version(name) for name in packages}
    run(sys.executable, "-u", str(runner), *args)

    records_dir = out / "records"
    records = list(records_dir.glob("*.json"))
    predictions = list(records_dir.glob("*.parquet"))
    if len(records) != 1 or predictions != [records[0].with_suffix(".parquet")]:
        raise RuntimeError("Expected exactly one JSON record and its matching Parquet file.")
    record_path, prediction_path = records[0], predictions[0]
    record = json.loads(record_path.read_text(encoding="utf-8"))
    expected = {
        "dms_id": protein, "split_scheme": "modulo", "test_fold": 0,
        "method": "lora_llr", "model_size": model_size, "n_total": 256, "seed": 42,
        "git_commit": commit,
        "prediction_sha256": hashlib.sha256(prediction_path.read_bytes()).hexdigest(),
    }
    if any(record.get(key) != value for key, value in expected.items()):
        raise RuntimeError("Record identity, Git SHA, or Parquet SHA-256 failed verification.")
    info.update(status="ok", task_id=record["task_id"], config_hash=record["config_hash"])
    artifacts = [record_path, prediction_path]
except Exception as error:
    info["error"] = f"{type(error).__name__}: {error}"
    print(info["error"], flush=True)
    with log.open("a", encoding="utf-8") as handle:
        traceback.print_exc(file=handle)

info["finished_utc"] = datetime.now(timezone.utc).isoformat()
info["elapsed_sec"] = round(time.monotonic() - started, 2)
metadata.write_text(json.dumps(info, indent=2) + "\n", encoding="utf-8")
zip_path = root / f"{root.name}_{info['status']}.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as bundle:
    for path in (log, metadata, *artifacts):
        name = f"records/{path.name}" if path in artifacts else path.name
        bundle.write(path, name)
print(f"{info['status']} | {info['elapsed_sec']}s | {zip_path}", flush=True)
files.download(str(zip_path))